# 03 — Evaluation

Loads the best IL checkpoint and runs chunk-safe inference (`run_inference()`) over every test instance, comparing the model's assignments against `GreedyLabeller` and reporting per-ULD weight/volume utilization plus a summary of packages assigned to `NONE`.  
**Run `01_setup.ipynb` first** so that `DATA_DIR`, `SAVE_DIR`, `TEST_DIR`, `TEST_META`, `IL_SAVE` are defined.

In [ ]:
import sys, os, collections
sys.path.insert(0, '..')

import pandas as pd
import torch
from tqdm.auto import tqdm

from src.config import DEVICE
from src.model import TransformerClusterer
from src.labeller import GreedyLabeller, DEFAULT_LABELLER
from src.data_utils import ClusteringDataset
from src.inference import run_inference

# These must be set — either run 01_setup.ipynb or define them here
# DATA_DIR    = '../good_data'
# SAVE_DIR    = '../clustering_v2'
# TEST_DIR    = os.path.join(DATA_DIR, 'synthetic_test')
# TEST_META   = os.path.join(TEST_DIR,  'metadata.csv')
# IL_SAVE     = os.path.join(SAVE_DIR, 'transformer_imitation_v2.pt')

In [ ]:
# Load the best model checkpoint
checkpoint = torch.load(IL_SAVE, map_location=DEVICE)
model = TransformerClusterer().to(DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from {IL_SAVE} "
      f"(val_loss: {checkpoint['val_loss']:.4f}, val_acc: {checkpoint['val_acc']:.1%})")

In [ ]:
# Prepare the test dataset (used only for its chunk index/grouping — raw dataframes are
# re-read per instance below so we always run inference on the FULL, unchunked instance).
val_ds_raw = ClusteringDataset(TEST_DIR, TEST_META, labeller=DEFAULT_LABELLER, device='cpu')

# Group dataset items by original instance tag
# This maps 'instance_001__uldchunk0_pkgchunk0' -> 'instance_001'
# And then groups all such items together to process the full instance once.
instance_tags = [item[0].split('__')[0] for item in val_ds_raw._index]
unique_instance_tags = sorted(list(set(instance_tags)))  # Sort for consistent order

# Overall counters for 'NONE' assignments
none_priority_total_model = 0
none_economy_total_model = 0

# Overall counters for comparison
model_none_labeller_uld_economy = 0

labeller = GreedyLabeller()  # Initialize GreedyLabeller for comparison

print("\n--- Analyzing ULD Capacity Utilization on Test Set (Aggregated per Instance) ---")

for tag in tqdm(unique_instance_tags, desc="Processing Full Instances"):
    u_path = os.path.join(val_ds_raw.data_dir, f'{tag}_ulds.csv')
    p_path = os.path.join(val_ds_raw.data_dir, f'{tag}_packages.csv')
    full_ulds_df = pd.read_csv(u_path)
    full_pkgs_df = pd.read_csv(p_path)

    # Run inference for the entire (potentially chunked) instance
    # run_inference already handles the internal chunking and stitching
    instance_assignment_model = run_inference(model, full_pkgs_df, full_ulds_df, device=DEVICE)
    instance_assignment_labeller = labeller.label(full_pkgs_df, full_ulds_df)

    # Initialize current occupied weight and volume for this instance's ULDs
    uld_occupied_weight = collections.defaultdict(float)
    uld_occupied_volume = collections.defaultdict(float)
    uld_priority_count = collections.defaultdict(int)
    uld_economy_count = collections.defaultdict(int)

    # Extract package weight, volume, and type for quick lookup
    pkg_info = full_pkgs_df.set_index('Package_ID').apply(
        lambda r: {
            'Weight': r['Weight'],
            'Volume': r['Length'] * r['Width'] * r['Height'],
            'Is_Priority': (r['Type'] == 'Priority')
        },
        axis=1
    ).to_dict()

    current_instance_none_priority_model = 0
    current_instance_none_economy_model = 0
    current_instance_model_none_labeller_uld_economy = 0

    # Populate occupied weight, volume, and package counts based on instance_assignment
    for pkg_id, assigned_uld_id_model in instance_assignment_model.items():
        if pkg_id in pkg_info:  # Ensure package exists in the current instance
            pkg_weight = pkg_info[pkg_id]['Weight']
            pkg_volume = pkg_info[pkg_id]['Volume']
            is_priority = pkg_info[pkg_id]['Is_Priority']
            assigned_uld_id_labeller = instance_assignment_labeller.get(pkg_id, 'UNKNOWN')

            if assigned_uld_id_model != 'NONE':
                uld_occupied_weight[assigned_uld_id_model] += pkg_weight
                uld_occupied_volume[assigned_uld_id_model] += pkg_volume
                if is_priority:
                    uld_priority_count[assigned_uld_id_model] += 1
                else:
                    uld_economy_count[assigned_uld_id_model] += 1
            else:
                if is_priority:
                    current_instance_none_priority_model += 1
                else:
                    current_instance_none_economy_model += 1

                # Check for model assigning to NONE when labeller assigns to ULD
                if not is_priority and assigned_uld_id_labeller != 'NONE':
                    current_instance_model_none_labeller_uld_economy += 1

    none_priority_total_model += current_instance_none_priority_model
    none_economy_total_model += current_instance_none_economy_model
    model_none_labeller_uld_economy += current_instance_model_none_labeller_uld_economy

    print(f"\nInstance: {tag}")
    print("  ULD | Wgt Occ (%) | Vol Occ (%) | Prio Pkgs | Eco Pkgs")
    print("  ----------------------------------------------------------------------------------")
    for _, uld_row in full_ulds_df.iterrows():  # Iterate over original full_ulds_df
        uld_id = uld_row['ULD_ID']
        weight_limit = uld_row['Weight_Limit']
        volume_limit = uld_row['Length'] * uld_row['Width'] * uld_row['Height']

        occupied_w = uld_occupied_weight.get(uld_id, 0.0)
        occupied_v = uld_occupied_volume.get(uld_id, 0.0)
        prio_pkgs = uld_priority_count.get(uld_id, 0)
        eco_pkgs = uld_economy_count.get(uld_id, 0)

        weight_pct = (occupied_w / weight_limit * 100) if weight_limit > 0 else 0
        volume_pct = (occupied_v / volume_limit * 100) if volume_limit > 0 else 0

        weight_status = "(OVER)" if occupied_w > weight_limit + 1e-6 else ""
        volume_status = "(OVER)" if occupied_v > volume_limit + 1e-6 else ""

        print(f"  {uld_id:<3} | {occupied_w:8.2f} ({weight_pct:6.2f}%) {weight_status:<6} | "
              f"{occupied_v:8.2f} ({volume_pct:6.2f}%) {volume_status:<6} | "
              f"{prio_pkgs:^9} | {eco_pkgs:^9}")

    if current_instance_none_priority_model > 0 or current_instance_none_economy_model > 0:
        print(f"  NONE|                                                         | "
              f"{current_instance_none_priority_model:^9} | {current_instance_none_economy_model:^9}")

print("\n--- Summary of Packages Assigned to NONE (across all instances) ---")
print(f"Total Priority Packages assigned to NONE (Model): {none_priority_total_model}")
print(f"Total Economy Packages assigned to NONE (Model):  {none_economy_total_model}")
print(f"Total Economy Packages assigned to NONE by Model, but ULD by GreedyLabeller: "
      f"{model_none_labeller_uld_economy}")